# 🔍 Deepfake Detection — Exploratory Data Analysis (EDA)

**Week 2 Presentation Notebook**

This notebook provides essential visual analysis of our preprocessed FaceForensics++ dataset:
1. Sample face images (real vs fake)
2. Class distribution
3. Image size verification (224×224)
4. Face detection success rate
5. Train/Val/Test split distribution

In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
from collections import Counter

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Paths (adjust if running from a different location)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
SPLITS_DIR = os.path.join(PROJECT_ROOT, 'data', 'splits')

print(f'Project root : {PROJECT_ROOT}')
print(f'Processed dir: {PROCESSED_DIR}')
print(f'Splits dir   : {SPLITS_DIR}')

## 1. Dataset Overview
Scan the processed directory to count videos and images.

In [ ]:
def count_video_folders(base_dir):
    """Count leaf directories that contain at least one .jpg file."""
    count = 0
    for root, dirs, files in os.walk(base_dir):
        if any(f.endswith('.jpg') for f in files):
            count += 1
    return count

def count_images(base_dir):
    """Count all .jpg files recursively."""
    return len(glob.glob(os.path.join(base_dir, '**', '*.jpg'), recursive=True))

real_dir = os.path.join(PROCESSED_DIR, 'real')
fake_dir = os.path.join(PROCESSED_DIR, 'fake')

real_videos = count_video_folders(real_dir)
fake_videos = count_video_folders(fake_dir)
real_images = count_images(real_dir)
fake_images = count_images(fake_dir)
total_images = real_images + fake_images

print('=' * 50)
print('  DATASET OVERVIEW')
print('=' * 50)
print(f'  Real videos : {real_videos}')
print(f'  Fake videos : {fake_videos}')
print(f'  Total videos: {real_videos + fake_videos}')
print(f'  Real images : {real_images:,}')
print(f'  Fake images : {fake_images:,}')
print(f'  Total images: {total_images:,}')
print('=' * 50)

## 2. Sample Face Images (6 Real + 6 Fake)
Display a 2×6 grid showing sample face crops.

In [ ]:
def get_random_images(base_dir, n=6, seed=42):
    """Get n random .jpg image paths from a directory tree."""
    all_imgs = glob.glob(os.path.join(base_dir, '**', '*.jpg'), recursive=True)
    random.seed(seed)
    if len(all_imgs) >= n:
        return random.sample(all_imgs, n)
    return all_imgs[:n]

real_samples = get_random_images(real_dir, 6)
fake_samples = get_random_images(fake_dir, 6)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Sample Face Crops: REAL (top) vs FAKE (bottom)', fontsize=16, fontweight='bold')

for i, img_path in enumerate(real_samples):
    img = mpimg.imread(img_path)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'REAL', fontsize=10, color='green', fontweight='bold')
    axes[0, i].axis('off')

for i, img_path in enumerate(fake_samples):
    img = mpimg.imread(img_path)
    axes[1, i].imshow(img)
    # Determine fake subcategory from path
    parts = img_path.replace('\\', '/').split('/')
    subcategory = ''
    for j, part in enumerate(parts):
        if part == 'fake' and j + 1 < len(parts):
            subcategory = parts[j+1]
            break
    label = f'FAKE ({subcategory})' if subcategory else 'FAKE'
    axes[1, i].set_title(label, fontsize=10, color='red', fontweight='bold')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'sample_faces.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/sample_faces.png')

## 3. Class Distribution
Bar chart showing the Real vs Fake image counts.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Video-level distribution ---
categories = ['Real', 'Fake']
video_counts = [real_videos, fake_videos]
colors = ['#2ecc71', '#e74c3c']

bars = ax1.bar(categories, video_counts, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
ax1.set_title('Video-Level Class Distribution', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Videos')
for bar, count in zip(bars, video_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(count), ha='center', fontweight='bold', fontsize=12)

# --- Image-level distribution ---
image_counts = [real_images, fake_images]

bars2 = ax2.bar(categories, image_counts, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
ax2.set_title('Image-Level Class Distribution', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Face Images')
for bar, count in zip(bars2, image_counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{count:,}', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/class_distribution.png')

## 4. Image Size Verification
Verify that all images are 224×224 pixels.

In [ ]:
# Sample a subset for efficiency (checking all ~32,000 would be slow in a notebook)
all_images = glob.glob(os.path.join(PROCESSED_DIR, '**', '*.jpg'), recursive=True)
sample_size = min(500, len(all_images))
random.seed(42)
sample_imgs = random.sample(all_images, sample_size)

sizes = []
non_standard = []

for img_path in sample_imgs:
    try:
        img = Image.open(img_path)
        w, h = img.size
        sizes.append((w, h))
        if (w, h) != (224, 224):
            non_standard.append((img_path, w, h))
    except Exception as e:
        non_standard.append((img_path, 'ERROR', str(e)))

size_counter = Counter(sizes)

print(f'Checked {sample_size} random images out of {len(all_images):,} total')
print(f'\nSize distribution:')
for size, count in size_counter.most_common():
    pct = count / sample_size * 100
    print(f'  {size[0]}×{size[1]} : {count} ({pct:.1f}%)')

if len(non_standard) == 0:
    print(f'\n✅ ALL {sample_size} sampled images are 224×224 — VERIFIED')
else:
    print(f'\n⚠️  {len(non_standard)} non-standard images found:')
    for item in non_standard[:5]:
        print(f'    {item}')

## 5. Face Detection Success Rate
Calculate how many frames successfully had faces detected.

In [ ]:
# Expected: 32 frames per video
FRAMES_PER_VIDEO = 32
total_videos_processed = real_videos + fake_videos
expected_total = total_videos_processed * FRAMES_PER_VIDEO

if expected_total > 0:
    success_rate = (total_images / expected_total) * 100
else:
    success_rate = 0.0

print(f'Total videos processed : {total_videos_processed}')
print(f'Frames per video       : {FRAMES_PER_VIDEO}')
print(f'Expected total faces   : {expected_total:,}')
print(f'Actual faces detected  : {total_images:,}')
print(f'Face detection rate    : {success_rate:.1f}%')

# Visual
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Detected', 'Missed'], [total_images, expected_total - total_images],
              color=['#2ecc71', '#e74c3c'], edgecolor='white', width=0.5)
ax.set_title(f'Face Detection Results ({success_rate:.1f}% success)', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Frames')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{int(bar.get_height()):,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'detection_rate.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Train / Val / Test Split Distribution
Visualize the split sizes and class balance within each split.

In [ ]:
# Load split CSVs
train_df = pd.read_csv(os.path.join(SPLITS_DIR, 'train.csv'))
val_df   = pd.read_csv(os.path.join(SPLITS_DIR, 'val.csv'))
test_df  = pd.read_csv(os.path.join(SPLITS_DIR, 'test.csv'))

print('CSV columns:', list(train_df.columns))
print(f'\nSplit sizes (video-level):')
print(f'  Train : {len(train_df)} videos')
print(f'  Val   : {len(val_df)} videos')
print(f'  Test  : {len(test_df)} videos')
print(f'  Total : {len(train_df) + len(val_df) + len(test_df)} videos')

print(f'\nClass balance per split:')
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    real_n = (df['label'] == 0).sum()
    fake_n = (df['label'] == 1).sum()
    print(f'  {name:5s}: real={real_n}, fake={fake_n}  ({real_n/(real_n+fake_n)*100:.1f}% real)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Split sizes ---
split_names = ['Train (70%)', 'Val (15%)', 'Test (15%)']
split_counts = [len(train_df), len(val_df), len(test_df)]
split_colors = ['#3498db', '#f39c12', '#9b59b6']

bars = ax1.bar(split_names, split_counts, color=split_colors, edgecolor='white', linewidth=1.5)
ax1.set_title('Split Sizes (Video Level)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Videos')
for bar, count in zip(bars, split_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             str(count), ha='center', fontweight='bold', fontsize=12)

# --- Stacked class balance per split ---
x = np.arange(3)
width = 0.5
real_counts = [(df['label'] == 0).sum() for df in [train_df, val_df, test_df]]
fake_counts = [(df['label'] == 1).sum() for df in [train_df, val_df, test_df]]

ax2.bar(x, real_counts, width, label='Real (0)', color='#2ecc71', edgecolor='white')
ax2.bar(x, fake_counts, width, bottom=real_counts, label='Fake (1)', color='#e74c3c', edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels(['Train', 'Val', 'Test'])
ax2.set_title('Class Balance per Split', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Videos')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'split_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/split_distribution.png')

In [ ]:
# Show first few rows of the train split
print('First 10 rows of train.csv:')
train_df.head(10)

## ✅ Summary

| Metric | Value |
|--------|-------|
| Total Videos | ~1,000 |
| Total Face Images | ~32,000 |
| Image Size | 224×224 |
| Face Detection Rate | See above |
| Split Ratio | 70/15/15 |
| Split Level | VIDEO (no data leakage) |
| Labels | 0=Real, 1=Fake |

**Week 2 Complete — Ready for Week 3 CNN Training!**